# 04. Statistical Hypothesis Testing, Effect Sizes & Inferential Rigor
**Project:** Telco Customer Churn & Retention Intelligence  
**Phase:** Phase 2 - Exploratory Data Analysis & Customer Behavior Analysis  
**Dataset:** `data/processed/telco_churn_cleaned.csv`

---

## Executive Overview & Statistical Objectives
Exploratory data analysis reveals patterns; inferential statistics tests whether those patterns reflect genuine underlying population dynamics or random sampling variation.

In this notebook, we execute:
1. **Chi-Square ($\chi^2$) Tests of Independence** across all 16 categorical features versus `Churn`.
2. **Cramér's V Effect Size Calculations** to standardize and rank feature association strength.
3. **Distributional Normality Diagnostics** for numerical features.
4. **Non-Parametric & Parametric Group Comparisons** (`Mann-Whitney U` and `Welch's t-test`).
5. **Standardized Effect Sizes (Cohen's d)** for numerical variables.
6. **Methodological Rigor & Critical Evaluation:**
   - Distinguishing **Correlation from Causation**.
   - Evaluating **Statistical Significance vs. Business Importance**.
   - Documenting **Analytical Limitations** (cross-sectional snapshot design).


In [1]:
import os
from pathlib import Path
import pandas as pd
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
import seaborn as sns

import matplotlib
matplotlib.use('Agg')
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['font.sans-serif'] = 'Arial'
plt.rcParams['figure.autolayout'] = True

DATA_PATH = (Path.cwd().parent / "data/processed/telco_churn_cleaned.csv" 
             if Path.cwd().name == "notebooks" 
             else Path("data/processed/telco_churn_cleaned.csv")).resolve()
FIGURES_DIR = (Path.cwd().parent / "reports/figures" 
               if Path.cwd().name == "notebooks" 
               else Path("reports/figures")).resolve()
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DATA_PATH)
print(f"Data loaded for statistical testing. Sample size: N = {len(df):,}")


Data loaded for statistical testing. Sample size: N = 7,043


## 1. Chi-Square ($\chi^2$) Tests of Independence & Cramér's V
For each categorical attribute, we test the null hypothesis:
- $H_0$: Customer churn is independent of feature $X$.
- $H_1$: Customer churn is statistically dependent on feature $X$.

Because large sample sizes ($N = 7,043$) routinely yield infinitesimal p-values ($p < 0.001$), we compute **Cramér's V** to evaluate practical effect size:
$$V = \sqrt{\frac{\chi^2}{N \cdot (\min(r, c) - 1)}}$$
- $V < 0.10$: Negligible association
- $0.10 \le V < 0.20$: Weak association
- $0.20 \le V < 0.30$: Moderate association
- $V \ge 0.30$: Strong association


In [2]:
cat_cols = [
    'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines',
    'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport',
    'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod'
]

chi2_results = []
n = len(df)

for col in cat_cols:
    contingency = pd.crosstab(df[col], df['Churn'])
    chi2, p_val, dof, _ = stats.chi2_contingency(contingency)
    r, c = contingency.shape
    cramers_v = np.sqrt(chi2 / (n * (min(r, c) - 1)))
    
    is_significant = p_val < 0.05
    strength = (
        "Strong" if cramers_v >= 0.30 else (
            "Moderate" if cramers_v >= 0.20 else (
                "Weak" if cramers_v >= 0.10 else "Negligible"
            )
        )
    )
    if not is_significant:
        strength = "Not Significant (p >= 0.05)"
        
    chi2_results.append({
        "Feature": col,
        "Chi2 Stat": round(chi2, 2),
        "dof": dof,
        "p-value": p_val,
        "p-val formatted": f"< 1e-10" if p_val < 1e-10 else f"{p_val:.4f}",
        "Cramer's V": round(cramers_v, 4),
        "Association Strength": strength
    })

chi2_df = pd.DataFrame(chi2_results).sort_values(by="Cramer's V", ascending=False)
print("--- Chi-Square Test Results & Effect Sizes Ranked ---")
print(chi2_df[['Feature', 'Chi2 Stat', 'dof', 'p-val formatted', "Cramer's V", 'Association Strength']].to_string(index=False))


--- Chi-Square Test Results & Effect Sizes Ranked ---
         Feature  Chi2 Stat  dof p-val formatted  Cramer's V        Association Strength
        Contract    1184.60    2         < 1e-10      0.4101                      Strong
  OnlineSecurity     850.00    2         < 1e-10      0.3474                      Strong
     TechSupport     828.20    2         < 1e-10      0.3429                      Strong
 InternetService     732.31    2         < 1e-10      0.3225                      Strong
   PaymentMethod     648.14    3         < 1e-10      0.3034                      Strong
    OnlineBackup     601.81    2         < 1e-10      0.2923                    Moderate
DeviceProtection     558.42    2         < 1e-10      0.2816                    Moderate
 StreamingMovies     375.66    2         < 1e-10      0.2310                    Moderate
     StreamingTV     374.20    2         < 1e-10      0.2305                    Moderate
PaperlessBilling     258.28    1         < 1e-10      0.

In [3]:
# Visualizing Cramér's V Rankings
fig, ax = plt.subplots(figsize=(12, 7))

colors = ['#d9534f' if v >= 0.30 else ('#f0ad4e' if v >= 0.15 else '#2b5c8f') for v in chi2_df["Cramer's V"]]
bars = ax.barh(chi2_df['Feature'][::-1], chi2_df["Cramer's V"][::-1], color=colors[::-1], edgecolor='black', height=0.6)
ax.set_title("Categorical Predictor Association with Churn (Cramér's V Effect Size)", fontsize=14, fontweight='bold', pad=12)
ax.set_xlabel("Cramér's V (Effect Size)", fontsize=12)
ax.set_xlim(0, 0.48)
ax.axvline(0.30, color='red', linestyle='--', label='Strong Threshold (V >= 0.30)')
ax.axvline(0.15, color='orange', linestyle=':', label='Moderate Threshold (V >= 0.15)')
ax.legend(loc='lower right', fontsize=11)

for bar in bars:
    w = bar.get_width()
    ax.text(w + 0.008, bar.get_y() + bar.get_height()/2., f"{w:.3f}", ha='left', va='center', fontweight='bold', fontsize=10)

fig1_path = FIGURES_DIR / "04_cramers_v_ranking.png"
plt.savefig(fig1_path, dpi=300, bbox_inches='tight')
plt.close(fig)
print(f"[Saved Figure] {fig1_path.resolve()}")


[Saved Figure] C:\Users\Sruthi\Documents\telco-churn-app\reports\figures\04_cramers_v_ranking.png


### Crucial Finding on Non-Significant Features
- **`gender`:** $\chi^2 = 0.48, p = 0.487$ (Fails statistical significance; $p > 0.05$). Churn behavior is completely independent of gender. Gender must NOT be used for churn targeting or risk scoring.
- **`PhoneService`:** $\chi^2 = 0.92, p = 0.339$ (Fails statistical significance; $p > 0.05$). Presence of standard landline service alone is not predictive of retention.


## 2. Numerical Features: Normality Diagnostics & Hypothesis Testing
We examine whether `tenure`, `MonthlyCharges`, and `TotalCharges` satisfy parametric normality assumptions.


In [4]:
num_cols = ['tenure', 'MonthlyCharges', 'TotalCharges']

# Normality testing using D'Agostino-Pearson test
normality_results = []
for c in num_cols:
    k2, p = stats.normaltest(df[c])
    normality_results.append({
        "Feature": c,
        "D'Agostino K2": round(k2, 2),
        "p-value": f"{p:.2e}",
        "Normal?": "No (p < 0.001)" if p < 0.001 else "Yes"
    })

print("--- Normality Diagnostics ---")
print(pd.DataFrame(normality_results).to_string(index=False))


--- Normality Diagnostics ---
       Feature  D'Agostino K2   p-value        Normal?
        tenure       76258.51  0.00e+00 No (p < 0.001)
MonthlyCharges       11419.53  0.00e+00 No (p < 0.001)
  TotalCharges         808.60 2.60e-176 No (p < 0.001)


Because all three numerical features violate Gaussian normality assumptions, we report:
1. **Primary Test:** Non-parametric **Mann-Whitney U Test** (Wilcoxon rank-sum test), which compares median ranks without distributional assumptions.
2. **Comparison Test:** Parametric **Welch's Two-Sample t-test** (accounting for unequal variances).
3. **Standardized Effect Size:** **Cohen's d**:
$$d = \frac{\bar{x}_1 - \bar{x}_2}{s_{pooled}}$$


In [5]:
num_tests = []

for c in num_cols:
    churned_vals = df[df['Churn'] == 'Yes'][c]
    retained_vals = df[df['Churn'] == 'No'][c]
    
    # 1. Mann-Whitney U test
    u_stat, u_p = stats.mannwhitneyu(churned_vals, retained_vals, alternative='two-sided')
    
    # 2. Welch's t-test
    t_stat, t_p = stats.ttest_ind(churned_vals, retained_vals, equal_var=False)
    
    # 3. Cohen's d
    n1, n2 = len(churned_vals), len(retained_vals)
    s1, s2 = churned_vals.std(), retained_vals.std()
    s_pooled = np.sqrt(((n1 - 1) * s1**2 + (n2 - 1) * s2**2) / (n1 + n2 - 2))
    cohen_d = (churned_vals.mean() - retained_vals.mean()) / s_pooled
    
    magnitude = (
        "Large" if abs(cohen_d) >= 0.8 else (
            "Medium" if abs(cohen_d) >= 0.5 else (
                "Small" if abs(cohen_d) >= 0.2 else "Negligible"
            )
        )
    )
    
    num_tests.append({
        "Feature": c,
        "Churned Mean": round(churned_vals.mean(), 2),
        "Retained Mean": round(retained_vals.mean(), 2),
        "Mann-Whitney U": int(u_stat),
        "U p-value": f"< 1e-10" if u_p < 1e-10 else f"{u_p:.4f}",
        "Welch t-stat": round(t_stat, 2),
        "Cohen's d": round(cohen_d, 3),
        "Effect Magnitude": magnitude
    })

num_test_df = pd.DataFrame(num_tests)
print("--- Numerical Hypothesis Testing & Effect Sizes ---")
print(num_test_df.to_string(index=False))


--- Numerical Hypothesis Testing & Effect Sizes ---
       Feature  Churned Mean  Retained Mean  Mann-Whitney U U p-value  Welch t-stat  Cohen's d Effect Magnitude
        tenure         17.98          37.57         2515538   < 1e-10        -34.82     -0.852            Large
MonthlyCharges         74.44          61.27         6003125   < 1e-10         18.41      0.446            Small
  TotalCharges       1531.80        2549.91         3381224   < 1e-10        -18.71     -0.458            Small


## 3. Methodological Discussion: Inferential Rigor
In a professional Data Analyst portfolio, demonstrating clear critical thinking and methodological rigor is paramount.

### A. Distinguishing Correlation from Causation
- **Contract Type & Churn ($V = 0.410$):** Month-to-month contracts strongly correlate with churn (42.7% vs. 2.8% on 2-year). However, **signing a month-to-month contract does not cause customer dissatisfaction**. Rather, customers with lower commitment, uncertain requirements, or price sensitivity select month-to-month plans to preserve flexibility. Forcing customers into long-term contracts without addressing core value propositions may depress acquisition rather than improve retention.
- **Payment Method ($V = 0.303$):** Electronic check payers churn at 45.3%. Electronic check payments require active monthly intervention and fail more often due to expired checking data. In contrast, credit card and bank transfer payers experience automatic billing inertia.

### B. Statistical Significance vs. Business Importance
- With $N = 7,043$, even minor deviations achieve extreme statistical significance ($p < 0.001$).
- For example, `MultipleLines` achieves $p = 0.00346$, but its Cramér's V is only **0.0401** (trivial practical effect).
- Conversely, `Contract` ($V = 0.4101$) and `OnlineSecurity` ($V = 0.3474$) possess massive practical and operational significance.

### C. Analytical Limitations of the Dataset
1. **Cross-Sectional Snapshot:** The dataset captures a single point in time rather than longitudinal monthly customer event logs. We cannot construct true historical cohort survival curves without monthly transition timestamps.
2. **Survival Bias:** Customers with 60–72 months tenure represent survivor accounts who have passed the high-hazard onboarding cliff.
3. **Absence of Customer Sentiment & Usage Metrics:** Key operational drivers such as customer support ticket counts, network outages, bandwidth utilization, and Net Promoter Score (NPS) are absent from this public dataset.
